This notebook is a basic tutorial that demonstrates how to configure a simulation using Concordia.

<a href="https://colab.research.google.com/github/google-deepmind/concordia/blob/main/examples/selling_cookies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title Imports

from collections.abc import Mapping, Sequence
from concordia.contrib import language_models as language_model_utils
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import entity as entity_lib
from concordia.typing import prefab as prefab_lib
from concordia.typing import scene as scene_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers


from concordia.contrib.language_models.openai.gpt_model import GptLanguageModel

/home/taesur/projects/certificate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:


from concordia.contrib.language_models.openai.gpt_model import GptLanguageModel
from concordia.language_model import language_model
from concordia.utils import measurements as measurements_lib

OPENROUTER_BASE = "https://openrouter.ai/api/v1"

class OpenRouterLanguageModel(GptLanguageModel):
    
    def __init__(
        self,
        model_name: str,
        *,
        api_key: str,
        measurements: measurements_lib.Measurements | None = None,
        channel: str = language_model.DEFAULT_STATS_CHANNEL
    ):
        super().__init__(
            model_name=model_name,
            api_key=api_key, 
            api_base=OPENROUTER_BASE, 
            measurements=measurements, 
            channel=channel
        )

In [29]:
# @title Language Model Selection: provide key or select DISABLE_LANGUAGE_MODEL

# By default this colab uses models via an external API so you must provide an
# API key. TogetherAI offers open weights models from all sources.

API_KEY = os.environ['OPENROUTER_API_KEY']  # @param {type: 'string'}
# See concordia/language_model/utils.py
API_TYPE = 'openai'  # e.g. 'together_ai' or 'openai'.
MODEL_NAME = (  # for API_TYPE = 'together_ai', we recommend MODEL_NAME = 'google/gemma-3-27b-it'
    'openai/gpt-4o'
)
# To debug without spending money on API calls, set DISABLE_LANGUAGE_MODEL=True
DISABLE_LANGUAGE_MODEL = False
OPENROUTER_BASE = "https://openrouter.ai/api/v1"

In [30]:
model = OpenRouterLanguageModel(
    model_name=MODEL_NAME,
    api_key=API_KEY
)

In [31]:
# @title Setup sentence encoder

if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7594.85it/s]


In [32]:
test = model.sample_text(
    'Is societal and technological progress like getting a clearer picture of '
    'something true and deep?'
)
print(test)

Societal and technological progress can be seen as processes that enhance our understanding and capabilities, much like focusing a lens to bring a picture into clearer view. Technological advancements provide us with tools and methods to gain deeper insights into complex phenomena, while societal progress often involves refining our values, institutions, and behaviors to better align with ideals such as justice, equality, and sustainability. Together, these forms of progress can help illuminate truths about the world and our place in it, though interpretations and directions of progress may differ across cultures and perspectives.


In [33]:
# @title Load prefabs from packages to make the specific palette to use here.

prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

In [34]:
# @title Print menu of prefabs

display.display(
    display.Markdown(helper_functions.print_pretty_prefabs(prefabs))
)

---
**`basic__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'randomize_choices': True, 'prefix_entity_name': True, 'observation_history_length': 1000000, 'situation_perception_history_length': 25, 'self_perception_history_length': 1000000, 'person_by_situation_history_length': 5}
)
```
---
**`basic_scripted__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'script': []}
)
```
---
**`basic_with_plan__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?" and building a plan based on the answers. It then tries to execute the plan.',
    params={'name': 'Alice', 'goal': '', 'force_time_horizon': False}
)
```
---
**`conversational__Entity`**:
```python
Entity(
    description='An entity that participates in conversations, aiming to create a dynamically balanced and engaging dialogue.',
    params={'name': 'Debra'}
)
```
---
**`fake_assistant_with_configurable_system_prompt__Entity`**:
```python
Entity(
    description='An entity that simulates an AI assistant with a configurable system prompt.',
    params={'name': 'Assistant', 'system_prompt': 'Assistant is a helpful and harmless AI assistant.'}
)
```
---
**`minimal__Entity`**:
```python
Entity(
    description='An entity that has a minimal set of components and is configurable by the user. The initial set of components manage memory, observations, and instructions. If goal is specified, the entity will have a goal constant component.',
    params={'name': 'Alice', 'goal': '', 'custom_instructions': '', 'extra_components': {}, 'extra_components_index': {}, 'randomize_choices': True}
)
```
---
**`puppet__Entity`**:
```python
Entity(
    description='An entity with fixed responses for specific calls to action.',
    params={'name': 'Puppet Agent', 'fixed_responses': {}, 'goal': ''}
)
```
---
**`rational__Entity`**:
```python
Entity(
    description='A rational agent that optimizes for its goal.',
    params={'name': 'Rational Agent', 'goal': '', 'randomize_choices': True, 'prefix_entity_name': True}
)
```
---
**`async_social_media__GameMaster`**:
```python
GameMaster(
    description='A game master for asynchronous social media simulations.',
    params={'name': 'forum_rules', 'forum_name': 'Community Forum', 'call_to_action': 'What does {name} do on the forum? Respond in JSON format with one of:\n{{"action": "post", "author": "{name}", "title": "...", "content": "..."}}\n{{"action": "reply", "author": "{name}", "post_id": "...", "content": "..."}}\n{{"action": "upvote", "author": "{name}", "post_id": "..."}}\n{{"action": "downvote", "author": "{name}", "post_id": "..."}}\n', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`async_social_media___NextActingEligiblePlayers`**:
```python
<concordia.prefabs.game_master.async_social_media._NextActingEligiblePlayers object at 0x7a4d0e2a5bb0>
```
---
**`dialogic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation.',
    params={'name': 'conversation rules', 'next_game_master_name': 'default rules', 'acting_order': 'game_master_choice', 'can_terminate_simulation': True}
)
```
---
**`dialogic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation. This game master is designed to be used with scenes.',
    params={'name': 'conversation rules', 'scenes': (), 'extra_components': {}, 'extra_components_index': {}, 'external_queue': None, 'allow_llm_fallback': True}
)
```
---
**`formative_memories_initializer__GameMaster`**:
```python
GameMaster(
    description='An initializer for all entities that generates formative memories from their childhood.',
    params={'name': 'initial setup rules', 'next_game_master_name': 'default rules', 'shared_memories': [], 'player_specific_context': {}, 'player_specific_memories': {}}
)
```
---
**`game_theoretic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling matrix game. decisions, designed to be used with scenes.',
    params={'name': 'decision rules', 'scenes': (), 'action_to_scores': <function _default_action_to_scores at 0x7a4dfc5a1300>, 'scores_to_observation': <function _default_scores_to_observation at 0x7a4dfc5a13a0>, 'external_queue': None}
)
```
---
**`generic__GameMaster`**:
```python
GameMaster(
    description='A general purpose game master.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'extra_components': {}, 'extra_components_index': {}, 'acting_order': 'game_master_choice'}
)
```
---
**`interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'verbose': False}
)
```
---
**`marketplace__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a marketplace.',
    params={'name': 'ExperimenterGM', 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`open_ended_interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'sequence_of_events': [], 'embedder': None, 'verbose': False}
)
```
---
**`physically_situated_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master for physical time/place simulations with scene support. Combines full world simulation with structured scene progressions.',
    params={'name': 'physical action rules', 'scenes': (), 'next_game_master_name': None, 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}, 'external_queue': None}
)
```
---
**`psychology_experiment__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a psychology experiment defined by custom observation and action specification components.',
    params={'name': 'ExperimenterGM', 'scenes': (), 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`scripted__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'ScriptedGM', 'script': [], 'verbose': False}
)
```
---
**`situated__GameMaster`**:
```python
GameMaster(
    description='A general game master for games set in a specific location.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`situated_in_time_and_place__GameMaster`**:
```python
GameMaster(
    description='A general game master for games situated in a physical time/place.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---

# Two Friends Catching Up

In [35]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='conversational__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Jessica',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='conversational__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Chloe',
            'conversation_style': (
                'Speaks with a lot of energy and excitement. Uses casual, '
                'modern language and slang common for a Southern California '
                'girl in their mid-20s.'
            ),
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dialogic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
            'next_game_master_name': 'conversation rules',
            'acting_order': 'fixed',
            'can_terminate_simulation': False,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'conversation rules',
            # Shared memory establishes the immediate context for their meeting.
            'shared_memories': [
                (
                    'Jessica and Chloe are best friends from college meeting at'
                    ' their favorite coffee shop to catch up after not seeing'
                    ' each other for months.'
                ),
            ],
            # Player-specific memories give them individual things to talk about.
            'player_specific_memories': {
                'Jessica': [
                    (
                        'Recently got a big promotion at her marketing job and'
                        ' is thinking about moving in with her boyfriend.'
                    ),
                ],
                'Chloe': [
                    (
                        'Just got back from a whirlwind backpacking trip'
                        ' through Europe and has tons of stories to share.'
                    ),
                ],
            },
        },
    ),
]

In [36]:
config = prefab_lib.Config(
    default_premise=(
        'Two people friends over coffee to discuss their lives and catch up'
    ),
    default_max_steps=20,
    prefabs=prefabs,
    instances=instances,
)

In [37]:
# @title Initialize the simulation
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [38]:
# @title Run the simulation
raw_log = []
results_log = runnable_simulation.play(
    max_steps=5,
    raw_log=raw_log,
)

Terminate? No
Game master: initial setup rules
Entity Jessica observed: Jessica and Chloe are best friends from college meeting at their favorite coffee shop to catch up after not seeing each other for months.


When Jessica was 5 years old, she experienced her first camping trip with her family. Tucked into her sleeping bag under a canopy of stars, she listened as her parents spun tales of woodland creatures and hidden worlds. The crisp night air and the crackling campfire fueled her vivid imagination, planting the seed for countless stories she would create. It was during this trip that Jessica realized the power of storytelling to transport her to magical realms, a discovery that ignited her lifelong passion for writing.






At 14, Jessica faced a daunting challenge when she was asked to read one of her stories aloud in front of her class. Nervous but resolute, she took to the podium, her voice trembling initially but gaining strength as she immersed herself in her narrative. The 

In [47]:
# @title Display the log
display.HTML(results_log.to_html())

# Love Island First Date

In [ ]:
PLAYER_ONE = 'Cody'
PLAYER_TWO = 'Megan'

instances = [
    prefab_lib.InstanceConfig(
        prefab='conversational__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': PLAYER_ONE,
            'conversation_style': (
                'Talk like a charismatic and flirty stud from Miami.'
            ),
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='conversational__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': PLAYER_TWO,
            'conversation_style': (
                'Talk like a stereotypical Love Island bombshell from Sussex.'
            ),
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dialogic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
            'next_game_master_name': 'conversation rules',
            'acting_order': 'fixed',
            'can_terminate_simulation': False,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'conversation rules',
            'player_specific_memories': {
                PLAYER_ONE: [
                    'Is a confident personal trainer from Miami, USA.',
                    'He is on a first date in the Love Island villa.',
                    'He thinks English girls are "unreal".',
                ],
                PLAYER_TWO: [
                    'Is a classic "bombshell" from Sussex.',
                    'She is on a first date in the Love Island villa.',
                    'She is not afraid to "step on toes" to find her man.',
                ],
            },
        },
    ),
]

config = prefab_lib.Config(
    default_premise=(
        'Night has fallen on the Love Island villa. The iconic fire pit is '
        f'lit, casting a warm glow. {PLAYER_ONE} and {PLAYER_TWO} are sat on '
        'the curved sofa around the fire, drinks in hand. The other islanders '
        'are out of earshot, giving them their first chance to properly graft.'
    ),
    default_max_steps=60,
    prefabs=prefabs,
    instances=instances,
)

In [ ]:
# @title Initialize the simulation
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [ ]:
# @title Run the simulation
raw_log = []
results_log = runnable_simulation.play(max_steps=10, raw_log=raw_log)

In [ ]:
# @title Display the log
display.HTML(results_log.to_html())

```
Copyright 2025 DeepMind Technologies Limited.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```